# SpAM Pilot Analysis

Descriptive analysis of pilot data collected via the SpAM (Spatial Arrangement Method) task.
Data directory: `data/pilot/` (local only, gitignored).

In [1]:
import warnings
import plotly.io as pio

from analysis.pilot.parser import load_pilot_data
from analysis.pilot.figures import (
    fig_completion_status,
    fig_trial_duration_per_subject,
    fig_moves_per_subject,
    fig_duration_progression,
    fig_moves_progression,
    fig_duration_vs_moves,
    fig_within_subject_variability,
    fig_demographics,
    fig_pairwise_distance_distribution,
)

pio.renderers.default = "browser"

## 0. Load data

In [2]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    data = load_pilot_data("data/pilot")

df_trials = data["trials"]
df_status = data["status"]

for w in caught_warnings:
    print(f"[{w.category.__name__}] {w.message}")

print(f"\nTrials dataframe: {df_trials.shape[0]} rows × {df_trials.shape[1]} cols")
print(df_status["completion_status"].value_counts().to_string())

[UserWarning] Participant 6150fbd25056424b64062835: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 69f5f8a6cd820e91eab3f3f7: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 698095549e3f340d0843b024: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 5d9e3ec9611b0b0017b14b9a: revoked consent (status=REJECTED), excluded from trials.
[UserWarning] Participant 69f5bb76f845c6ae7f522328: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 60001f74b9d8d70009041d15: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 67e069199f5b036960b0cc5f: revoked consent (status=RETURNED), excluded from trials.
[UserWarning] Participant 694555b4beadda60a5901ec0: APPROVED but no session file found (erroneous completion), excluded from trials.
[UserWarning] Participant 697cb3

## 1. Completion status

In [3]:
fig_completion_status(df_status).show()

## 2. Trial duration per subject

In [4]:
fig_trial_duration_per_subject(df_trials).show()

## 3. Number of moves per subject

In [5]:
fig_moves_per_subject(df_trials).show()

## 4. Trial duration over task progression

In [6]:
fig_duration_progression(df_trials).show()

## 5. Moves over task progression

In [7]:
fig_moves_progression(df_trials).show()

## 6. Trial duration vs. number of moves

In [8]:
fig_duration_vs_moves(df_trials).show()

## 7. Within-subject variability and reliability

In [9]:
fig_within_subject_variability(df_trials).show()

## 8. Participant demographics

In [10]:
fig_demographics(df_trials).show()

## 9. Pairwise distance distribution

In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

from analysis.pilot.simulate_null_distances import simulate as _sim_null

# Shared null: same K as images_per_trial, enough trials for a stable reference
null_distances = _sim_null(num_dots=20, num_trials=1000, seed=42)
print(f"Null: {len(null_distances):,} distances  mean={null_distances.mean():.3f}  sd={null_distances.std():.3f}")

fig_pairwise_distance_distribution(df_trials, null_distribution=null_distances).show()

Null: 190,000 distances  mean=0.369  sd=0.175


In [ ]:
from analysis.pilot.figures import fig_ks_distance_per_subject

# null_distances defined in the cell above (shared across cohorts)
fig_ks_distance_per_subject(df_trials, null_distances).show()

## 10. Temporal engagement

Do subjects work throughout the trial, or front-load moves and sit idle?

- **Left**: cumulative move fraction vs. time fraction within trial. Diagonal = uniform activity; curve bending top-left = front-loading.
- **Right**: average move rate (moves/s, 5 s bins) over absolute time. Dashed line = 60 s (V2 timer unlock).

In [15]:
from analysis.pilot.figures import fig_move_temporal_profile

fig_move_temporal_profile(df_trials).show()

## 11. Cohort comparisons

Pairwise Mann-Whitney U tests between task versions, on per-subject aggregates (one value
per subject, avoiding pseudo-replication across trials). With more than two cohorts present,
every pairwise comparison within a metric is run and p-values are corrected across that
family (default Bonferroni; configurable via `mwu_family(..., correction=...)` — any method
string accepted by `statsmodels.stats.multitest.multipletests` works, e.g. `"holm"`, `"fdr_bh"`).

Five comparisons:
1. Trial duration (s)
2. Moves per trial
3. SNR (σ_d / mean\|Δd\|) — **v1 vs v2 only**, see note before that cell
4. KS distance from null — per-trial D, averaged per subject
5. Idle tail fraction — (RT − t_last_move) / RT

Effect size is rank-biserial *r* = 1 − 2U / (n_A × n_B); \|r\| ≥ 0.5 is conventionally large.

In [ ]:
summary = (
    df_trials
    .assign(
        rt_s=df_trials["rt"] / 1000,
        qc_flag_int=df_trials["qc_flag"].astype(int),
    )
    .groupby("task_version")
    .agg(
        n_subjects=("participant_id",  "nunique"),
        n_trials=("trial_number",      "count"),
        rt_s_mean=("rt_s",             "mean"),
        rt_s_median=("rt_s",           "median"),
        rt_s_sd=("rt_s",               "std"),
        rt_s_min=("rt_s",              "min"),
        rt_s_max=("rt_s",              "max"),
        moves_mean=("n_moves",         "mean"),
        moves_median=("n_moves",       "median"),
        moves_sd=("n_moves",           "std"),
        qc_flag_rate=("qc_flag_int",   "mean"),
    )
    .T
    .rename(columns=lambda v: f"v{v:g}")
    .round(2)
)
summary

In [ ]:
import json
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


def mwu_compare(df, value_col, group_a, group_b, group_col="task_version"):
    """Pairwise Mann-Whitney U between two specified groups (no group-count assumption)."""
    a = df.loc[df[group_col] == group_a, value_col].dropna().astype(float).values
    b = df.loc[df[group_col] == group_b, value_col].dropna().astype(float).values
    U, p = mannwhitneyu(a, b, alternative="two-sided")
    r = 1 - 2 * U / (len(a) * len(b))
    return {
        "comparison":  f"v{group_a:g} vs v{group_b:g}",
        "N (A)":       len(a),                              "N (B)":       len(b),
        "median (A)":  round(float(np.median(a)), 3),       "median (B)":  round(float(np.median(b)), 3),
        "mean (A)":    round(float(np.mean(a)), 3),         "mean (B)":    round(float(np.mean(b)), 3),
        "std (A)":     round(float(np.std(a, ddof=1)), 3),  "std (B)":     round(float(np.std(b, ddof=1)), 3),
        "U":           round(U, 1),
        "p_raw":       p,
        "r":           round(r, 3),
    }


def mwu_family(df, value_col, group_col="task_version", correction="bonferroni", alpha=0.05):
    """
    Pairwise Mann-Whitney U for every pair of groups in group_col, with a multiple-
    comparison correction applied across the family. `correction` accepts any method
    string supported by statsmodels.stats.multitest.multipletests, e.g. "bonferroni",
    "holm", "fdr_bh", "fdr_by", "sidak", "holm-sidak", ...
    """
    groups = sorted(df[group_col].dropna().unique())
    rows = [mwu_compare(df, value_col, a, b, group_col) for a, b in combinations(groups, 2)]
    result = pd.DataFrame(rows).set_index("comparison")
    _, p_corrected, _, _ = multipletests(result["p_raw"], alpha=alpha, method=correction)
    result["p_raw"] = result["p_raw"].round(4)
    result["p_corrected"] = p_corrected.round(4)
    return result[["N (A)", "N (B)", "median (A)", "median (B)", "mean (A)", "mean (B)",
                    "std (A)", "std (B)", "U", "p_corrected", "p_raw", "r"]]


def print_bottom_line(result, alpha=0.05):
    """One-line plain-English summary per pairwise comparison."""
    for comparison, row in result.iterrows():
        sig = "significant" if row["p_corrected"] < alpha else "not significant"
        print(f"  {comparison}: U={row['U']:.1f}  p_corrected={row['p_corrected']:.4f} ({sig}, "
              f"p_raw={row['p_raw']:.4f})  r={row['r']:.3f}")

### Test 1 — Trial duration

In [ ]:
subj_rt = (
    df_trials.assign(rt_s=df_trials["rt"] / 1000)
    .groupby(["participant_id", "task_version"])["rt_s"]
    .mean()
    .reset_index()
)

result_rt = mwu_family(subj_rt, "rt_s", correction="bonferroni")
print_bottom_line(result_rt)
result_rt

### Test 2 — Moves per trial

In [ ]:
subj_moves = (
    df_trials
    .groupby(["participant_id", "task_version"])["n_moves"]
    .mean()
    .reset_index()
)

result_moves = mwu_family(subj_moves, "n_moves", correction="bonferroni")
print_bottom_line(result_moves)
result_moves

### Test 3 — SNR (v1 vs v2 only)

**Why v3 is excluded, not generalized:** v1/v2 reliability is measured from individual image
pairs that incidentally recur across distinct trials (a side effect of the old
`unique_images_per_subject` design — different surrounding images each time). v3 instead
repeats whole trials verbatim (`is_trial_repeat` / `repeat_of_trial_number`), holding context
constant. These two measures convolve different sources of variance — v1/v2 includes context
effects, v3 is closer to pure response noise — so they are not comparable in absolute
magnitude (see the subtitle on the within-subject variability figure above). v3 is therefore
dropped here rather than silently mixed into the v1-vs-v2 comparison.

**For future versions:** if a later version (v4+) also uses the trial-repeat mechanism, add a
separate **Test 3′** comparing it against v3 using the trial-repeat measure
(`_repeated_trial_distances` / `_reliability_pair_distances` in `figures.py`) — do not extend
*this* test to include it.

In [ ]:
from collections import defaultdict

from analysis.pilot.parser import parse_pairwise_distances


def _subject_snr(df_s):
    """v1/v2-only reliability measure: incidentally-repeated image pairs (see note above)."""
    pair_obs = defaultdict(list)
    for pw_json in df_s["pairwise_distances"]:
        for pair, dist in parse_pairwise_distances(pw_json).items():
            pair_obs[pair].append(dist)
    d1, d2 = [], []
    for obs in pair_obs.values():
        if len(obs) >= 2:
            d1.append(obs[0])
            d2.append(obs[1])
    if not d1:
        return np.nan
    all_dists = [d for pw_json in df_s["pairwise_distances"]
                 for d in parse_pairwise_distances(pw_json).values()]
    sigma_d = float(np.std(all_dists)) if len(all_dists) > 1 else 1.0
    mean_abs_diff = float(np.mean(np.abs(np.array(d1) - np.array(d2))))
    return sigma_d / mean_abs_diff if mean_abs_diff > 0 else np.nan


df_trials_v1v2 = df_trials[df_trials["task_version"].isin([1.0, 2.0])]

subj_snr = (
    df_trials_v1v2
    .groupby(["participant_id", "task_version"])
    .apply(_subject_snr, include_groups=False)
    .reset_index()
    .rename(columns={0: "snr"})
)

result_snr = mwu_family(subj_snr, "snr", correction="bonferroni")
print_bottom_line(result_snr)
result_snr

### Test 4 — KS distance from null

D is computed per trial against the shared null (see section 9), then averaged per subject —
the same aggregation as `fig_ks_distance_per_subject` above.

In [ ]:
from analysis.pilot.figures import trial_ks_distance

# null_distances defined in the figures section above (section 9)
df_trials_ks = df_trials.assign(
    ks_D=df_trials["pairwise_distances"].apply(trial_ks_distance, null_distribution=null_distances)
)
subj_ks = (
    df_trials_ks
    .groupby(["participant_id", "task_version"])["ks_D"]
    .mean()
    .reset_index()
)

result_ks = mwu_family(subj_ks, "ks_D", correction="bonferroni")
print_bottom_line(result_ks)
result_ks

### Test 5 — Idle tail fraction

In [ ]:
def _idle_tail_fraction(row):
    """Fraction of trial time after the last move: (RT - t_last) / RT."""
    try:
        moves = json.loads(row["moves"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    ts = [m["t"] for m in moves if isinstance(m.get("t"), (int, float))]
    if not ts or pd.isna(row["rt"]) or row["rt"] <= 0:
        return np.nan
    return (row["rt"] - max(ts)) / row["rt"]


subj_idle = (
    df_trials
    .assign(idle_tail=df_trials.apply(_idle_tail_fraction, axis=1))
    .groupby(["participant_id", "task_version"])["idle_tail"]
    .mean()
    .reset_index()
)

result_idle = mwu_family(subj_idle, "idle_tail", correction="bonferroni")
print_bottom_line(result_idle)
result_idle

## 12. Quality Control

Validate the catch-trial QC thresholds in `SpAM_Task/task_config.json` against actual collected
data. `computeCatchQcFlag` (`utils.js`) flags a catch trial if **any** of:

- `cluster_mean_distance > catch_trials.cluster_max_mean` (0.15) — images not clustered tightly
- `cluster_sd > catch_trials.cluster_max_sd` (0.10) — cluster too spread (SD of pairwise distances; not a stored field, recomputed below)
- `max_dist_to_target > catch_trials.location_tolerance` (0.20) — worst-case per-image distance to the target corner/centre (mirrors `allImagesNearTarget`; not a stored field, recomputed below)

`cluster_mean_distance` is a stored field; `cluster_sd` and `max_dist_to_target` are recomputed
here from `pairwise_distances` / `final_locations` since the task only saves the boolean QC
outcome, not these intermediate values.

**Note**: the `qc_flag` column reflects whichever thresholds were active in `task_config.json`
*at the time each session ran* — which may differ from the current values above (config has
changed over time). The recomputed metrics below are compared against **today's** thresholds,
which is what answers "are these values still valid going forward."

In [ ]:
df_catch = data["catch_trials"]

# Current catch_trials QC thresholds (SpAM_Task/task_config.json)
_CATCH_THRESH = {
    "cluster_mean_distance": 0.15,   # catch_trials.cluster_max_mean
    "cluster_sd":            0.10,   # catch_trials.cluster_max_sd
    "max_dist_to_target":    0.20,   # catch_trials.location_tolerance
}

_EDGE = 0.15  # fraction from edge for corner targets (utils.js _targetPoint)
_TARGET_FRAC = {
    "center":              (0.50, 0.50),
    "top left corner":     (_EDGE, _EDGE),
    "top right corner":    (1 - _EDGE, _EDGE),
    "bottom left corner":  (_EDGE, 1 - _EDGE),
    "bottom right corner": (1 - _EDGE, 1 - _EDGE),
}


def _cluster_sd(pw_json):
    """Sample SD (ddof=1) of a trial's pairwise distances -- mirrors computeSD() in utils.js.
    Generic across catch and main trials (same pairwise_distances schema)."""
    dists = list(parse_pairwise_distances(pw_json).values())
    return float(np.std(dists, ddof=1)) if len(dists) > 1 else 0.0


def _max_dist_to_target(row):
    """Worst-case (max) per-image normalised distance to the catch target -- mirrors allImagesNearTarget() in utils.js."""
    try:
        locs = json.loads(row["final_locations"])
    except (json.JSONDecodeError, TypeError):
        return np.nan
    if not locs:
        return np.nan
    fx, fy = _TARGET_FRAC.get(row["catch_trial_target_location"], (0.5, 0.5))
    w, h = row["sort_area_width"], row["sort_area_height"]
    diag = np.sqrt(w**2 + h**2)
    dists = [
        np.sqrt((loc["x"] * w - fx * w) ** 2 + (loc["y"] * h - fy * h) ** 2) / diag
        for loc in locs
    ]
    return max(dists)


df_catch = df_catch.assign(
    cluster_sd=df_catch["pairwise_distances"].apply(_cluster_sd),
    max_dist_to_target=df_catch.apply(_max_dist_to_target, axis=1),
)


def _qc_row(df, thresh_map, direction="max"):
    """
    Per-cohort QC summary row.

    direction="max": metric is capped (flag if value > thresh); shows median/p90/max
                      -- the upper tail is what's relevant for headroom.
    direction="min": metric is floored (flag if value < thresh); shows median/p10/min
                      -- the lower tail is what's relevant for headroom.
    """
    out = {}
    for metric, thresh in thresh_map.items():
        vals = df[metric].dropna()
        if direction == "max":
            out[(metric, "median")]               = round(float(vals.median()), 3)
            out[(metric, "p90")]                  = round(float(vals.quantile(0.9)), 3)
            out[(metric, "max")]                  = round(float(vals.max()), 3)
            out[(metric, f"flagged (>{thresh})")] = round(float((vals > thresh).mean()), 3)
        else:
            out[(metric, "median")]                = round(float(vals.median()), 3)
            out[(metric, "p10")]                   = round(float(vals.quantile(0.1)), 3)
            out[(metric, "min")]                   = round(float(vals.min()), 3)
            out[(metric, f"flagged (<{thresh})")]  = round(float((vals < thresh).mean()), 3)
    out[("recorded", "qc_flag rate")] = round(float(df["qc_flag"].mean()), 3)
    out[("recorded", "n_trials")]     = len(df)
    return pd.Series(out)


rows = {
    f"v{v:g}": _qc_row(df_catch[df_catch["task_version"] == v], _CATCH_THRESH)
    for v in sorted(df_catch["task_version"].unique())
}
rows["all"] = _qc_row(df_catch, _CATCH_THRESH)

qc_table = pd.DataFrame(rows).T
qc_table.columns = pd.MultiIndex.from_tuples(qc_table.columns)
qc_table

### Experimental (main) trial thresholds

`computeMainQcFlag` (`utils.js`) flags a main trial if **either**:

- `pairwise_sd < quality_control.min_pairwise_distance_sd` (0.04) — images piled in one spot
- `move_ratio < quality_control.min_move_item_ratio` (0.75), where `move_ratio = n_moves / n_items` — too few moves relative to the number of images on screen

`quality_control.min_trial_rt_ms` (60 s) is excluded here since it's UI-enforced (the Done button
is disabled until the floor elapses), not a post-hoc statistical flag.

Both remaining checks are **floors**, the opposite direction from the catch-trial checks above, so
the table below shows the lower tail (median / p10 / min) and flags values *below* threshold.

In [ ]:
# Current main-trial QC thresholds (SpAM_Task/task_config.json)
_MAIN_THRESH = {
    "pairwise_sd": 0.04,   # quality_control.min_pairwise_distance_sd
    "move_ratio":  0.75,   # quality_control.min_move_item_ratio
}


def _n_items(final_locations_json):
    try:
        return len(json.loads(final_locations_json))
    except (json.JSONDecodeError, TypeError):
        return np.nan


df_trials_qc = df_trials.assign(
    pairwise_sd=df_trials["pairwise_distances"].apply(_cluster_sd),
    n_items=df_trials["final_locations"].apply(_n_items),
)
df_trials_qc["move_ratio"] = df_trials_qc["n_moves"] / df_trials_qc["n_items"]

rows = {
    f"v{v:g}": _qc_row(df_trials_qc[df_trials_qc["task_version"] == v], _MAIN_THRESH, direction="min")
    for v in sorted(df_trials_qc["task_version"].unique())
}
rows["all"] = _qc_row(df_trials_qc, _MAIN_THRESH, direction="min")

qc_table_main = pd.DataFrame(rows).T
qc_table_main.columns = pd.MultiIndex.from_tuples(qc_table_main.columns)
qc_table_main